In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!lscpu | grep -E "Model name|CPU\(s\)|cache" || true
!apt-get install -y -qq openmpi-bin libopenmpi-dev
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-9164454a-b1aa-8838-3a8d-2b3564f221ec, 7.5, 15360 MiB
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
L1d cache:                               32 KiB (1 instance)
L1i cache:                               32 KiB (1 instance)
L2 cache:                                1 MiB (1 instance)
L3 cache:                                38.5 MiB (1 instance)
NUMA node0 CPU(s):                       0,1


In [3]:
GITHUB_USERNAME = "Nurspyun"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "Set GITHUB_USERNAME to your real GitHub handle before running."
)
print(f"Verified solo student handle: {GITHUB_USERNAME}")

Verified solo student handle: Nurspyun


In [4]:
%%writefile hybrid_mpi_cuda_cache.py
import os
import sys
import re
import json
import time
import hashlib
import platform
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, njit, float64, uint64

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2
GLOBAL_NX = 512
GLOBAL_NY = 512
LOCAL_N = 256
PAD_N = LOCAL_N + 2
SEG = LOCAL_N
STENCIL_STEPS = 300
SHM_ITERS = 40_000_000
SHM_SLOTS = 64
SHM_REPEATS = 2
LAMBDA_DAMP = 0.15
DT_SAFETY = 0.9
DX = 1.0 / (GLOBAL_NX + 1)
CHECKPOINTS = (0, 100, 200, 300)
STRIDES = (1, 8, 16)


def get_gpu_uuid():
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def detect_cpu_line_size():
    path = "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    return 64


def detect_cpu_model():
    try:
        with open("/proc/cpuinfo", "r", encoding="utf-8") as f:
            for line in f:
                if line.lower().startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return "unknown"


def init_hybrid_context(github_user):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Requires exactly 4 MPI ranks (2x2 grid), got {size}.")
    cart_comm = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    cart_rank = cart_comm.Get_rank()
    coords = cart_comm.Get_coords(cart_rank)
    up, down = cart_comm.Shift(0, 1)
    left, right = cart_comm.Shift(1, 1)
    cuda.select_device(0)
    gpu_uuid = comm.bcast(get_gpu_uuid() if rank == 0 else None, root=0)
    cpu_sig = f"{platform.machine()}::{os.cpu_count()}::{detect_cpu_model()}"
    line_bytes = detect_cpu_line_size()
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}::{cpu_sig}".encode("utf-8")
    ).hexdigest()
    seed = int(digest[:16], 16)
    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    node_rank = node_comm.Get_rank()
    if node_comm.Get_size() != 4:
        raise RuntimeError("All 4 ranks must share one node.")
    nbytes = (SHM_SLOTS + 8) * 8 if node_rank == 0 else 0
    win = MPI.Win.Allocate_shared(nbytes, 8, comm=node_comm)
    buf, _ = win.Shared_query(0)
    raw = np.frombuffer(buf, dtype=np.uint64, count=SHM_SLOTS + 8)
    addr = int(raw.ctypes.data)
    off = ((64 - addr % 64) % 64) // 8
    shm_view = raw[off:off + SHM_SLOTS]
    node_comm.Barrier()
    return {
        "comm": comm,
        "rank": cart_rank,
        "cart_comm": cart_comm,
        "coords": coords,
        "up": up,
        "down": down,
        "left": left,
        "right": right,
        "gpu_uuid": gpu_uuid,
        "cpu_sig": cpu_sig,
        "line_bytes": line_bytes,
        "seed": seed,
        "seed_hex": f"{seed:016x}",
        "github_user": github_user.strip(),
        "node_comm": node_comm,
        "node_rank": node_rank,
        "win": win,
        "shm_view": shm_view,
        "shm_base": int(shm_view.ctypes.data),
    }


def initialize_fields(ctx):
    cy, cx = ctx["coords"]
    idx = np.arange(PAD_N, dtype=np.float64)
    y = (cy * LOCAL_N + idx) * DX
    x = (cx * LOCAL_N + idx) * DX
    Y, X = np.meshgrid(y, x, indexing="ij")
    u = np.exp(-((X - 0.5) ** 2 + (Y - 0.5) ** 2) / 0.02)
    kap = 0.2 + 0.8 * np.exp(-((X - 0.7) ** 2 + (Y - 0.3) ** 2) / 0.05)
    src = 0.05 * np.exp(-((X - 0.25) ** 2 + (Y - 0.75) ** 2) / 0.01)
    u[0, :] = 0.0
    u[-1, :] = 0.0
    u[:, 0] = 0.0
    u[:, -1] = 0.0
    return (
        np.ascontiguousarray(u, dtype=np.float64),
        np.ascontiguousarray(kap, dtype=np.float64),
        np.ascontiguousarray(src, dtype=np.float64),
    )


@cuda.jit
def pack_halos_kernel(u, buf):
    t = cuda.grid(1)
    if t < 4 * SEG:
        seg = t // SEG
        k = t - seg * SEG
        if seg == 0:
            v = u[1, 1 + k]
        elif seg == 1:
            v = u[LOCAL_N, 1 + k]
        elif seg == 2:
            v = u[1 + k, 1]
        else:
            v = u[1 + k, LOCAL_N]
        buf[t] = v


@cuda.jit
def unpack_halos_kernel(u, buf):
    t = cuda.grid(1)
    if t < 4 * SEG:
        seg = t // SEG
        k = t - seg * SEG
        v = buf[t]
        if seg == 0:
            u[0, 1 + k] = v
        elif seg == 1:
            u[LOCAL_N + 1, 1 + k] = v
        elif seg == 2:
            u[1 + k, 0] = v
        else:
            u[1 + k, LOCAL_N + 1] = v


@cuda.jit
def tiled_stencil_kernel(u_in, u_out, kap, src, dt, inv_dx2, damp, i0, i1, j0, j1):
    tile = cuda.shared.array((SHARED_DIM, SHARED_DIM), float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    j = j0 + cuda.blockIdx.x * TILE_DIM + tx
    i = i0 + cuda.blockIdx.y * TILE_DIM + ty
    if i <= i1 + 1 and j <= j1 + 1:
        tile[ty + 1, tx + 1] = u_in[i, j]
        if ty == 0:
            tile[0, tx + 1] = u_in[i - 1, j]
        if ty == TILE_DIM - 1 and i <= i1:
            tile[SHARED_DIM - 1, tx + 1] = u_in[i + 1, j]
        if tx == 0:
            tile[ty + 1, 0] = u_in[i, j - 1]
        if tx == TILE_DIM - 1 and j <= j1:
            tile[ty + 1, SHARED_DIM - 1] = u_in[i, j + 1]
    cuda.syncthreads()
    if i >= i0 and i <= i1 and j >= j0 and j <= j1:
        c = tile[ty + 1, tx + 1]
        n = tile[ty, tx + 1]
        s = tile[ty + 2, tx + 1]
        w = tile[ty + 1, tx]
        e = tile[ty + 1, tx + 2]
        kc = kap[i, j]
        ke = 0.5 * (kc + kap[i, j + 1])
        kw = 0.5 * (kc + kap[i, j - 1])
        kn = 0.5 * (kc + kap[i - 1, j])
        ks = 0.5 * (kc + kap[i + 1, j])
        div = ke * (e - c) - kw * (c - w) + ks * (s - c) - kn * (c - n)
        u_out[i, j] = c + dt * (div * inv_dx2 + src[i, j] - damp * c)


def make_state(u0, kap, src):
    h_send = cuda.pinned_array(4 * SEG, dtype=np.float64)
    h_recv = cuda.pinned_array(4 * SEG, dtype=np.float64)
    h_send[:] = 0.0
    h_recv[:] = 0.0
    return {
        "u_in": cuda.to_device(u0),
        "u_out": cuda.to_device(u0.copy()),
        "kap": cuda.to_device(kap),
        "src": cuda.to_device(src),
        "d_send": cuda.device_array(4 * SEG, dtype=np.float64),
        "d_recv": cuda.to_device(np.zeros(4 * SEG, dtype=np.float64)),
        "h_send": h_send,
        "h_recv": h_recv,
        "s_int": cuda.stream(),
        "s_bnd": cuda.stream(),
    }


def launch_region(stream, st, p, i0, i1, j0, j1):
    gx = (j1 - j0) // TILE_DIM + 1
    gy = (i1 - i0) // TILE_DIM + 1
    tiled_stencil_kernel[(gx, gy), (TILE_DIM, TILE_DIM), stream](
        st["u_in"], st["u_out"], st["kap"], st["src"],
        p["dt"], p["inv_dx2"], p["damp"], i0, i1, j0, j1,
    )


def mpi_exchange(ctx, st):
    cc = ctx["cart_comm"]
    hs = st["h_send"]
    hr = st["h_recv"]
    reqs = [
        cc.Irecv(hr[0:SEG], source=ctx["up"], tag=11),
        cc.Irecv(hr[SEG:2 * SEG], source=ctx["down"], tag=10),
        cc.Irecv(hr[2 * SEG:3 * SEG], source=ctx["left"], tag=13),
        cc.Irecv(hr[3 * SEG:4 * SEG], source=ctx["right"], tag=12),
        cc.Isend(hs[0:SEG], dest=ctx["up"], tag=10),
        cc.Isend(hs[SEG:2 * SEG], dest=ctx["down"], tag=11),
        cc.Isend(hs[2 * SEG:3 * SEG], dest=ctx["left"], tag=12),
        cc.Isend(hs[3 * SEG:4 * SEG], dest=ctx["right"], tag=13),
    ]
    MPI.Request.Waitall(reqs)


def stage_halos(ctx, st, stream):
    pack_halos_kernel[4, SEG, stream](st["u_in"], st["d_send"])
    st["d_send"].copy_to_host(st["h_send"], stream=stream)
    stream.synchronize()
    mpi_exchange(ctx, st)
    st["d_recv"].copy_to_device(st["h_recv"], stream=stream)
    unpack_halos_kernel[4, SEG, stream](st["u_in"], st["d_recv"])


def step_serialized(ctx, st, p):
    s = st["s_bnd"]
    stage_halos(ctx, st, s)
    launch_region(s, st, p, 1, LOCAL_N, 1, LOCAL_N)
    s.synchronize()
    st["u_in"], st["u_out"] = st["u_out"], st["u_in"]


def step_overlapped(ctx, st, p):
    si = st["s_int"]
    sb = st["s_bnd"]
    launch_region(si, st, p, 2, LOCAL_N - 1, 2, LOCAL_N - 1)
    stage_halos(ctx, st, sb)
    launch_region(sb, st, p, 1, 1, 1, LOCAL_N)
    launch_region(sb, st, p, LOCAL_N, LOCAL_N, 1, LOCAL_N)
    launch_region(sb, st, p, 2, LOCAL_N - 1, 1, 1)
    launch_region(sb, st, p, 2, LOCAL_N - 1, LOCAL_N, LOCAL_N)
    sb.synchronize()
    si.synchronize()
    st["u_in"], st["u_out"] = st["u_out"], st["u_in"]


def global_energy(ctx, st):
    u_h = st["u_in"].copy_to_host()
    local = float(np.sum(u_h[1:LOCAL_N + 1, 1:LOCAL_N + 1] ** 2)) * DX * DX
    return ctx["cart_comm"].allreduce(local, op=MPI.SUM)


def run_simulation(ctx, mode, u0, kap, src, p, steps, record):
    st = make_state(u0, kap, src)
    step_fn = step_serialized if mode == "serialized" else step_overlapped
    energies = {}
    elapsed = 0.0
    ctx["cart_comm"].Barrier()
    for n in range(steps + 1):
        if record and n in CHECKPOINTS:
            energies[n] = global_energy(ctx, st)
        if n == steps:
            break
        t0 = time.perf_counter()
        step_fn(ctx, st, p)
        elapsed += time.perf_counter() - t0
    final = st["u_in"].copy_to_host()
    return elapsed * 1000.0, energies, final


def audit_ptx():
    asm = tiled_stencil_kernel.inspect_asm()
    ptx = next(iter(asm.values())) if isinstance(asm, dict) else asm
    return {
        "st_shared_f64": len(re.findall(r"st\.shared\.f64", ptx)),
        "ld_shared_f64": len(re.findall(r"ld\.shared\.f64", ptx)),
        "bar_sync": len(re.findall(r"bar\.sync", ptx)),
    }


@njit(nogil=True)
def host_shm_accumulate(shm, idx, probe, iters):
    acc = np.uint64(0)
    for _ in range(iters):
        shm[idx] = shm[idx] + np.uint64(1)
        acc += shm[probe]
    return acc


def audit_llvm():
    sig = host_shm_accumulate.signatures[0]
    ir = host_shm_accumulate.inspect_llvm(sig)
    chunks = ir.split("\ndefine ")
    body = [c for c in chunks if "host_shm_accumulate" in c.split("\n", 1)[0] and "cpython" not in c.split("\n", 1)[0]]
    text = body[0] if body else ir
    return {
        "scope": "core_function" if body else "full_module",
        "store_i64": len(re.findall(r"\bstore i64\b", text)),
        "load_i64": len(re.findall(r"\bload i64\b", text)),
        "branches": len(re.findall(r"\bbr (?:i1|label)\b", text)),
    }


def cache_line_tables(ctx):
    base = ctx["shm_base"]
    table = {}
    for stride in STRIDES:
        offsets = [r * stride for r in range(4)]
        cpu_lines = [(base + 8 * k) // 64 for k in offsets]
        gpu_sectors = [(base + 8 * k) // 32 for k in offsets]
        table[str(stride)] = {
            "slot_indices": offsets,
            "byte_spacing": stride * 8,
            "cpu_lines": [int(v) for v in cpu_lines],
            "distinct_cpu_lines": len(set(cpu_lines)),
            "distinct_gpu_sectors": len(set(gpu_sectors)),
        }
    return table


def benchmark_shm_strides(ctx):
    nc = ctx["node_comm"]
    nr = ctx["node_rank"]
    shm = ctx["shm_view"]
    host_shm_accumulate(shm, nr * 16, nr * 16, 1)
    nc.Barrier()
    results = {}
    for stride in STRIDES:
        idx = nr * stride
        best = None
        ok = True
        for _ in range(SHM_REPEATS):
            nc.Barrier()
            if nr == 0:
                shm[:] = 0
            nc.Barrier()
            t0 = time.perf_counter()
            acc = host_shm_accumulate(shm, idx, idx, SHM_ITERS)
            nc.Barrier()
            ms = (time.perf_counter() - t0) * 1000.0
            worst = nc.allreduce(ms, op=MPI.MAX)
            ok = ok and (int(shm[idx]) == SHM_ITERS)
            best = worst if best is None else min(best, worst)
        all_ok = nc.allreduce(1 if ok else 0, op=MPI.MIN) == 1
        results[str(stride)] = {"ms": float(best), "counter_verified": bool(all_ok)}
    return results


def coalescing_model():
    elems = LOCAL_N
    useful = elems * 8
    return {
        "column_elements": elems,
        "row_stride_bytes": PAD_N * 8,
        "useful_bytes": useful,
        "gpu_32B_sector_bytes_strided": elems * 32,
        "gpu_128B_line_bytes_strided": elems * 128,
        "cpu_64B_line_bytes_strided": elems * 64,
        "gpu_sector_amplification": (elems * 32) / useful,
        "cpu_line_amplification": (elems * 64) / useful,
        "packed_contiguous_bytes": useful,
        "packed_sectors": useful // 32,
        "bytes_avoided_per_column_gpu_sector_model": elems * 32 - useful,
        "bytes_avoided_per_exchange_two_columns": 2 * (elems * 32 - useful),
    }


def emit(rank, text):
    if rank == 0:
        print(text, flush=True)


def main():
    github_user = sys.argv[1]
    ctx = init_hybrid_context(github_user)
    cc = ctx["cart_comm"]
    rank = ctx["rank"]
    emit(rank, f"GITHUB_USERNAME={ctx['github_user']} INSTANCE_SEED=0x{ctx['seed_hex']}")
    emit(rank, f"GPU_UUID={ctx['gpu_uuid']} CPU_SIG={ctx['cpu_sig']} CPU_LINE_BYTES={ctx['line_bytes']}")
    emit(rank, f"SHM_BASE_MOD_64={ctx['shm_base'] % 64}")

    u0, kap, src = initialize_fields(ctx)
    kappa_max = cc.allreduce(float(kap[1:-1, 1:-1].max()), op=MPI.MAX)
    dt_max = DX * DX / (4.0 * kappa_max)
    dt = DT_SAFETY * dt_max
    p = {"dt": float(dt), "inv_dx2": float(1.0 / (DX * DX)), "damp": LAMBDA_DAMP}
    emit(rank, f"kappa_max={kappa_max:.12f} dx={DX:.12e} dt_max={dt_max:.12e} dt={dt:.12e}")

    for mode in ("serialized", "overlapped"):
        run_simulation(ctx, mode, u0, kap, src, p, 3, False)

    ms_s, e_s, f_s = run_simulation(ctx, "serialized", u0, kap, src, p, STENCIL_STEPS, True)
    ms_o, e_o, f_o = run_simulation(ctx, "overlapped", u0, kap, src, p, STENCIL_STEPS, True)

    local_diff = float(np.max(np.abs(f_s[1:-1, 1:-1] - f_o[1:-1, 1:-1])))
    max_diff = cc.allreduce(local_diff, op=MPI.MAX)
    gpu_ser = cc.allreduce(ms_s, op=MPI.MAX)
    gpu_ovl = cc.allreduce(ms_o, op=MPI.MAX)

    cfl_ok = dt <= dt_max
    parity_ok = max_diff < 1e-12
    energy_match = all(abs(e_s[k] - e_o[k]) <= 1e-12 * max(1.0, abs(e_s[k])) for k in CHECKPOINTS)
    emit(rank, f"gpu_serialized_ms={gpu_ser:.4f} gpu_overlapped_ms={gpu_ovl:.4f} overlap_speedup={gpu_ser / gpu_ovl:.4f}")
    emit(rank, f"max_diff={max_diff:.3e} parity_ok={parity_ok} cfl_ok={cfl_ok} energy_match={energy_match}")
    for k in CHECKPOINTS:
        emit(rank, f"step={k} global_energy_serialized={e_s[k]:.15e} global_energy_overlapped={e_o[k]:.15e}")
    if not (parity_ok and cfl_ok and energy_match):
        raise RuntimeError("Cross-rank verification failed.")

    ptx = audit_ptx()
    emit(rank, f"PTX_AUDIT {ptx}")

    shm = benchmark_shm_strides(ctx)
    llvm = audit_llvm()
    lines = cache_line_tables(ctx)
    emit(rank, f"LLVM_AUDIT {llvm}")
    for s in STRIDES:
        emit(rank, f"stride={s} ({s * 8}B) latency_ms={shm[str(s)]['ms']:.4f} distinct_cpu_lines={lines[str(s)]['distinct_cpu_lines']} verified={shm[str(s)]['counter_verified']}")
    ratio_false = shm["1"]["ms"] / shm["8"]["ms"]
    ratio_pad = shm["16"]["ms"] / shm["8"]["ms"]
    emit(rank, f"false_sharing_slowdown_stride1_over_stride8={ratio_false:.4f} stride16_over_stride8={ratio_pad:.4f}")

    if rank == 0:
        manifest = {
            "github_username": ctx["github_user"],
            "instance_seed": f"0x{ctx['seed_hex']}",
            "gpu_uuid": ctx["gpu_uuid"],
            "cpu_signature": ctx["cpu_sig"],
            "cpu_line_bytes": ctx["line_bytes"],
            "mpi_ranks": 4,
            "global_interior": [GLOBAL_NY, GLOBAL_NX],
            "local_padded": [PAD_N, PAD_N],
            "stencil_steps": STENCIL_STEPS,
            "shm_iters": SHM_ITERS,
            "kappa_max": kappa_max,
            "dx": DX,
            "dt_max": dt_max,
            "dt": dt,
            "cfl_ratio": dt / dt_max,
            "cfl_satisfied": bool(cfl_ok),
            "lambda_damp": LAMBDA_DAMP,
            "gpu_serialized_ms": gpu_ser,
            "gpu_overlapped_ms": gpu_ovl,
            "overlap_speedup": gpu_ser / gpu_ovl,
            "max_diff": max_diff,
            "parity_ok": bool(parity_ok),
            "energies_serialized": {str(k): v for k, v in e_s.items()},
            "energies_overlapped": {str(k): v for k, v in e_o.items()},
            "energy_relative_change_0_to_300": (e_s[300] - e_s[0]) / e_s[0],
            "ptx_audit": ptx,
            "llvm_audit": llvm,
            "shm_latency_ms": {k: v["ms"] for k, v in shm.items()},
            "shm_counter_verified": {k: v["counter_verified"] for k, v in shm.items()},
            "false_sharing_slowdown_stride1_over_stride8": ratio_false,
            "padding_ratio_stride16_over_stride8": ratio_pad,
            "shm_base_mod_64": ctx["shm_base"] % 64,
            "cache_line_tables": lines,
            "coalescing_model": coalescing_model(),
        }
        digest = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode("utf-8")).hexdigest()
        manifest["manifest_sha256"] = digest
        with open("hybrid_lab_manifest.json", "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)
        print("MANIFEST_WRITTEN hybrid_lab_manifest.json sha256=" + digest, flush=True)


if __name__ == "__main__":
    main()


Writing hybrid_mpi_cuda_cache.py


In [7]:
import re

src = open("hybrid_mpi_cuda_cache.py", encoding="utf-8").read()

new_func = r'''def _llvm_counts(text):
    return {
        "store_i64": len(re.findall(r"\bstore i64\b", text)),
        "load_i64": len(re.findall(r"\bload i64\b", text)),
        "phi_nodes": len(re.findall(r"\bphi\b", text)),
        "branches": len(re.findall(r"\bbr (?:i1|label)\b", text)),
    }


def audit_llvm():
    sig = host_shm_accumulate.signatures[0]
    ir = host_shm_accumulate.inspect_llvm(sig)
    chunks = ir.split("\ndefine ")[1:]
    named = [c for c in chunks if "host_shm_accumulate" in c.split("\n", 1)[0]]
    named.sort(key=len, reverse=True)
    text = named[0] if named else ir
    scope = "core_function"
    counts = _llvm_counts(text)
    if counts["store_i64"] == 0 or counts["load_i64"] == 0:
        counts = _llvm_counts(ir)
        scope = "full_module"
    counts["scope"] = scope
    counts["loop_store_load_present"] = (
        counts["store_i64"] > 0 and counts["load_i64"] > 0 and counts["branches"] > 0
    )
    return counts'''

src = re.sub(
    r"def audit_llvm\(\):.*?\n\n\ndef cache_line_tables",
    lambda m: new_func + "\n\n\ndef cache_line_tables",
    src,
    flags=re.S,
)
open("hybrid_mpi_cuda_cache.py", "w", encoding="utf-8").write(src)
print("def _llvm_counts" in src, "loop_store_load_present" in src)

True True


In [8]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "hybrid_mpi_cuda_cache.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Hybrid MPI-CUDA execution failed with code {proc.returncode}")


GITHUB_USERNAME=Nurspyun INSTANCE_SEED=0x87ff89d5b2086eca
GPU_UUID=GPU-9164454a-b1aa-8838-3a8d-2b3564f221ec CPU_SIG=x86_64::2::Intel(R) Xeon(R) CPU @ 2.00GHz CPU_LINE_BYTES=64
SHM_BASE_MOD_64=0
kappa_max=0.999998784052 dx=1.949317738791e-03 dt_max=9.499610667948e-07 dt=8.549649601153e-07
gpu_serialized_ms=933.4256 gpu_overlapped_ms=1794.2918 overlap_speedup=0.5202
max_diff=0.000e+00 parity_ok=True cfl_ok=True energy_match=True
step=0 global_energy_serialized=3.141592653579141e-02 global_energy_overlapped=3.141592653579141e-02
step=100 global_energy_serialized=3.120958531249748e-02 global_energy_overlapped=3.120958531249748e-02
step=200 global_energy_serialized=3.100627713551013e-02 global_energy_overlapped=3.100627713551013e-02
step=300 global_energy_serialized=3.080592406545054e-02 global_energy_overlapped=3.080592406545054e-02
PTX_AUDIT {'st_shared_f64': 5, 'ld_shared_f64': 5, 'bar_sync': 1}
LLVM_AUDIT {'store_i64': 2, 'load_i64': 3, 'phi_nodes': 4, 'branches': 16, 'scope': 'core_fun

In [10]:
import json
m = json.load(open("hybrid_lab_manifest.json"))
print("PTX:", m["ptx_audit"])
print("LLVM:", m["llvm_audit"])
print("serialized ms:", m["gpu_serialized_ms"], "overlapped ms:", m["gpu_overlapped_ms"], "speedup:", m["overlap_speedup"])
print("shm ms:", m["shm_latency_ms"])
print("false sharing ratio 1/8:", m["false_sharing_slowdown_stride1_over_stride8"])
print("padding ratio 16/8:", m["padding_ratio_stride16_over_stride8"])
print("kappa_max:", m["kappa_max"], "dx:", m["dx"], "dt_max:", m["dt_max"], "dt:", m["dt"], "cfl_ratio:", m["cfl_ratio"])
print("energies:", m["energies_serialized"])
print("energy relative change:", m["energy_relative_change_0_to_300"])
print("max_diff:", m["max_diff"])
print("coalescing:", m["coalescing_model"])
print("lines:", m["cache_line_tables"])

PTX: {'st_shared_f64': 5, 'ld_shared_f64': 5, 'bar_sync': 1}
LLVM: {'store_i64': 2, 'load_i64': 3, 'phi_nodes': 4, 'branches': 16, 'scope': 'core_function', 'loop_store_load_present': True}
serialized ms: 933.4256480012755 overlapped ms: 1794.2917979983122 speedup: 0.5202195367791307
shm ms: {'1': 295.3869369998756, '8': 94.75809000014124, '16': 82.94793999993999}
false sharing ratio 1/8: 3.1172740712633122
padding ratio 16/8: 0.8753652590487667
kappa_max: 0.9999987840522371 dx: 0.001949317738791423 dt_max: 9.499610667947604e-07 dt: 8.549649601152844e-07 cfl_ratio: 0.9
energies: {'0': 0.031415926535791415, '100': 0.031209585312497484, '200': 0.03100627713551013, '300': 0.030805924065450536}
energy relative change: -0.019416981690669476
max_diff: 0.0
coalescing: {'column_elements': 256, 'row_stride_bytes': 2064, 'useful_bytes': 2048, 'gpu_32B_sector_bytes_strided': 8192, 'gpu_128B_line_bytes_strided': 32768, 'cpu_64B_line_bytes_strided': 16384, 'gpu_sector_amplification': 4.0, 'cpu_line

### Analytical & Forensic Engineering Responses

- **Question 1 (GPU Coalescing vs. CPU Cache-Line Granularity):** The column u[1:257, 1] has 256 float64 elements separated by the row stride of 258 × 8 = 2,064 bytes, so every element lies in a different memory line. A 64-byte CPU cache line fetches 64 B to use 8 B: 256 × 64 = 16,384 B moved for 2,048 B useful (8.0× amplification), and the hardware prefetcher cannot help because the access is not sequential. On the T4 the L2 serves 32-byte sectors, so the same column touches 256 sectors = 8,192 B for 2,048 B useful (4.0× amplification; 32,768 B if whole 128-byte L1 lines are counted); a warp reading it issues 32 separate transactions instead of one coalesced request. pack_halos_kernel gathers the column into a contiguous 2,048 B buffer (64 sectors, 1.0× amplification), so one contiguous D2H DMA replaces 256 scattered 8-byte accesses. Bytes avoided per column: 8,192 − 2,048 = 6,144 B; per exchange (left + right columns): 12,288 B. The top/bottom rows are already contiguous and need no special treatment.

- **Question 2 (Shared-Memory Tile Synchronization & PTX Audit):** tiled_stencil_kernel.inspect_asm() reports st.shared.f64 = 5, ld.shared.f64 = 5 and bar.sync = 1. The five stores are the center cell plus the four halo edges (top row, bottom row, left column, right column) of the 18 × 18 tile; the five loads are the center and its four stencil neighbors; the single bar.sync is the cuda.syncthreads() barrier between the cooperative store phase and the load phase. A 16 × 16 block has 256 threads = 8 warps. Without the barrier, one warp can read tile[ty, tx+1], tile[ty+2, tx+1], tile[ty+1, tx] or tile[ty+1, tx+2] before the warp that owns that cell (or the halo) has written it. This is a Read-After-Write hazard: the reader gets stale or uninitialized shared memory, the stencil output becomes nondeterministic, and the serialized/overlapped parity check (max_diff = 0.000e+00 < 1e-12) would fail.

- **Question 3 (Dual-Stream Compute-Communication Overlap on Shared GPU Silicon):** Measured gpu_serialized_ms = 933.4256 and gpu_overlapped_ms = 1794.2918, a speedup of 0.5202, so the overlapped mode took about 1.92× longer than the serialized mode. (An earlier run gave 771.0 ms vs 934.9 ms, so the exact value varies between runs, but overlap never won.) Four MPI processes share one physical T4: each process owns a separate CUDA context and the driver time-slices the contexts instead of running them concurrently, so the deep-interior kernel of one rank competes with the kernels and copies of the other three. All four ranks also share one PCIe link and its copy engines, so their pinned D2H/H2D transfers queue behind each other, and the host has only 2 vCPUs for 4 MPI processes that also busy-wait in Waitall and stream synchronization. The overlap that stream_interior and stream_boundary could provide is therefore consumed by context switching and queueing, plus extra launches (1 interior + 4 border kernels instead of 1). With four dedicated GPUs each rank would have its own SMs, copy engines and PCIe lane, and with GPUDirect RDMA the NIC would read and write GPU memory directly without staging through pinned host buffers, so communication could genuinely hide behind the interior kernel. Both modes produced identical fields (max_diff = 0.000e+00).

- **Question 4 (MPI-3 Shared-Memory MESI RFO Ping-Pong):** Measured latencies for 40,000,000 updates: stride 1 (8 B) = 295.3869 ms, stride 8 (64 B) = 94.7581 ms, stride 16 (128 B) = 82.9479 ms. The false-sharing slowdown (stride 1 / stride 8) is 3.1173×, about 7.4 ns per update versus 2.4 ns. With stride 1 the four counters share one 64-byte line (distinct_cpu_lines = 1); with stride 8 and 16 each rank has its own line (distinct_cpu_lines = 4). The LLVM IR audit of host_shm_accumulate shows load i64 = 3, store i64 = 2 and 16 branches, so the read-modify-write stays in the loop and is not hoisted into a register (the final counters were verified = 40,000,000). MESI trace: Rank 0 writes shm_view[0]: core 0 issues a Read-For-Ownership and the line goes I→E→M in core 0 (I in the others). Rank 1 writes shm_view[1], which is in the same line: core 1's RFO invalidates core 0's copy, the dirty line is forwarded/written back and core 0 goes M→I while core 1 goes I→M. Rank 0's next write finds its copy in I, issues a new RFO, core 1 goes M→I and core 0 goes I→M again. The line ping-pongs between the cores on every update even though the ranks touch different 8-byte words. With stride 8 each line stays in M (or E) in its owner's cache and no invalidations occur. The slowdown is moderate because this Colab VM has only 2 vCPUs, so at most two ranks run truly simultaneously (the other two are time-sliced).

- **Question 5 (128-Byte Spatial Prefetcher Pairing & Padding Trade-Offs):** stride 8 (64 B padding) = 94.7581 ms and stride 16 (128 B padding) = 82.9479 ms, a ratio of 0.8754, so 128-byte padding was about 12% faster. The adjacent-cache-line (spatial) prefetcher treats an even/odd pair of 64-byte lines as one 128-byte aligned sector and tends to fetch the buddy line together with the demanded one. My cache-line table shows why this matters: with stride 8 the four counters occupy consecutive lines ending in ...373, ...374, ...375, ...376, so the lines 374 and 375 (ranks 1 and 2) form one even/odd 128-byte pair, and the prefetcher can pull a neighbor's line into a core and later cause invalidations. With stride 16 the lines are ...373, ...375, ...377, ...379, each in a different 128-byte pair, so this interaction disappears. The difference is small and may be partly timing noise on a shared 2-vCPU VM, so I treat it as consistent with, not proof of, prefetcher pairing. The cost of excess padding is capacity: each 8-byte counter occupies 128 B (16× waste instead of 8×), so with many counters the working set grows, evicts useful data from L1/L2 and causes capacity misses. For only four counters this footprint is negligible, so padding to 128 B is free here.

- **Question 6 (CFL Stability & Distributed Energy Dissipation):** kappa_max = 0.999998784052 and Δx = 1/513 = 1.949317738791e-03. Δt_max = Δx² / (4 · κ_max) = 9.499610667948e-07. The simulation uses Δt = 0.9 · Δt_max = 8.549649601153e-07, so Δt/Δt_max = 0.9 < 1 and the Courant condition is satisfied. Global energy E = Σ u² Δx² from cart_comm.allreduce: step 0 = 3.141592653579141e-02, step 100 = 3.120958531249748e-02, step 200 = 3.100627713551013e-02, step 300 = 3.080592406545054e-02, a relative decrease of 1.9417% over 300 steps. The energy decreases monotonically because the diffusion smooths the field and the damping term −λu (λ = 0.15) removes energy, only slightly offset by the weak source term; serialized and overlapped modes give identical values at every checkpoint.